# Data gathering and Preprocessing

## Introduction
Data is gathered from Kaggle using their API, and is a combination of several datasets.

- World happiness report
- World employment data
- World economic data

### Contents
1. Data Gathering
2. Data Cleaning
3. Combining Data


## 0. Importing Libraries

In [32]:
# Import necessary libraries
import pandas as pd
import numpy as np
import kagglehub
from kaggle.api.kaggle_api_extended import KaggleApi
from pathlib import Path

## 1. Data Gathering
Authenticate with Kaggle API and download the datasets.

In [33]:
from dotenv import load_dotenv
import os

load_dotenv()
kaggle_username = os.getenv('KAGGLE_USERNAME')
kaggle_key = os.getenv('KAGGLE_KEY')

api = KaggleApi()
api.authenticate()

# Step 1: Define Kaggle datasets to fetch
datasets = {
    "gdp_dataset": "zgrcemta/world-gdpgdp-gdp-per-capita-and-annual-growths",
    "world_happiness_dataset": "unsdsn/world-happiness",
    "unemployment_dataset": "pantanjali/unemployment-dataset"
}

In [34]:

# Check if the datasets are already downloaded
if not Path("data").exists():


    # Create a data folder to store datasets
    data_path = Path("data")
    data_path.mkdir(exist_ok=True)

    # Step 2: Download datasets
    for name, dataset in datasets.items():
        api.dataset_download_files(dataset, path=str(data_path / name), unzip=True)

    # Step 3: Delete gpd.csv, gdp_ppp.csv and gdp_growth file from dataset
    gdp_path = data_path / "gdp_dataset"
    gdp_path.joinpath("gdp.csv").unlink()
    gdp_path.joinpath("gdp_ppp.csv").unlink()
    gdp_path.joinpath("gdp_growth.csv").unlink()
    gdp_path.joinpath("gdp_ppp_per_capita.csv").unlink()
else:
    print("Data already downloaded")
    data_path = Path("data")



Dataset URL: https://www.kaggle.com/datasets/zgrcemta/world-gdpgdp-gdp-per-capita-and-annual-growths
Dataset URL: https://www.kaggle.com/datasets/unsdsn/world-happiness
Dataset URL: https://www.kaggle.com/datasets/pantanjali/unemployment-dataset


## 2. Data Cleaning
For each dataset:
- Remove unnecessary columns
- Remove rows with missing values



### 2.0 Load Data

In [35]:
datasets_loaded = {}
for name in datasets.keys():
    dataset_dir = data_path / name
    csv_files = list(dataset_dir.glob("*.csv"))
    datasets_loaded[name] = {csv_file.stem: pd.read_csv(csv_file) for csv_file in csv_files}

### 2.1 World Happiness Data

In [36]:
# See if happiness_data already exists
if not Path("data/happiness_data.csv").exists():
    world_happiness = datasets_loaded["world_happiness_dataset"]
    
    # Remove all columns except for 'Country' and happiness score
    world_happiness['2015'] = world_happiness['2015'][['Country', 'Happiness Score']]
    world_happiness['2016'] = world_happiness['2016'][['Country', 'Happiness Score']]
    world_happiness['2017'] = world_happiness['2017'][['Country', 'Happiness.Score']]
    world_happiness['2018'] = world_happiness['2018'][['Country or region', 'Score']]
    world_happiness['2019'] = world_happiness['2019'][['Country or region', 'Score']]
    
    # Rename the columns to be consistent
    world_happiness['2015'].rename(columns={'Happiness Score': 'Happiness_Score_2015'}, inplace=True)
    world_happiness['2016'].rename(columns={'Happiness Score': 'Happiness_Score_2016'}, inplace=True)
    world_happiness['2017'].rename(columns={'Happiness.Score': 'Happiness_Score_2017'}, inplace=True)
    world_happiness['2018'].rename(columns={'Country or region': 'Country', 'Score': 'Happiness_Score_2018'}, inplace=True)
    world_happiness['2019'].rename(columns={'Country or region': 'Country', 'Score': 'Happiness_Score_2019'}, inplace=True)
    
    # Merge the datasets on the 'Country' column
    happiness_data = pd.merge(world_happiness['2015'], world_happiness['2016'], on='Country', how='inner')
    happiness_data = pd.merge(happiness_data, world_happiness['2017'], on='Country', how='inner')
    happiness_data = pd.merge(happiness_data, world_happiness['2018'], on='Country', how='inner')
    happiness_data = pd.merge(happiness_data, world_happiness['2019'], on='Country', how='inner')
    
    # Calculate the mean happiness score across the years and add as a new column
    happiness_data['Happiness_Score_Mean'] = happiness_data[[
        'Happiness_Score_2015', 
        'Happiness_Score_2016', 
        'Happiness_Score_2017', 
        'Happiness_Score_2018', 
        'Happiness_Score_2019'
    ]].mean(axis=1)
    
    # Save to file
    happiness_data.to_csv('data/happiness_data.csv', index=False)
else:
    happiness_data = pd.read_csv('data/happiness_data.csv')

### 2.2 World Unemployment Data

In [37]:
# See if unemployment_data already exists
if not Path('data/unemployment_data.csv').exists():
    unemployment = datasets_loaded["unemployment_dataset"]['unemployment analysis']
    # Keep only the columns we need and rename to the universal index: Country
    unemployment = unemployment[['Country Name', '2015', '2016', '2017', '2018', '2019']]
    # Rename all yearly columns to Unemployment_Rate_x
    unemployment = unemployment.rename(columns={
        'Country Name': 'Country',
        '2015': 'Unemployment_Rate_2015', 
        '2016': 'Unemployment_Rate_2016', 
        '2017': 'Unemployment_Rate_2017', 
        '2018': 'Unemployment_Rate_2018', 
        '2019': 'Unemployment_Rate_2019'
    })

    # Calculate the mean unemployment rate across the years and add as a new column
    unemployment['Unemployment_Rate_Mean'] = unemployment[[
        'Unemployment_Rate_2015', 
        'Unemployment_Rate_2016', 
        'Unemployment_Rate_2017', 
        'Unemployment_Rate_2018', 
        'Unemployment_Rate_2019'
    ]].mean(axis=1)

    # Save the final dataset
    unemployment.to_csv('data/unemployment_data.csv', index=False)
else:
    unemployment = pd.read_csv('data/unemployment_data.csv')

### 2.3 World Economic Data

In [38]:

if not Path('data/gdp_data.csv').exists():
    # Load the GDP datasets
    gdp = datasets_loaded['gdp_dataset']['gdp_per_capita']
    gdp_growth = datasets_loaded['gdp_dataset']['gdp_per_capita_growth']
    
    # Keep only the columns we need and rename to the index: Country
    gdp = gdp[['Country Name', '2015', '2016', '2017', '2018', '2019']].rename(columns={'Country Name': 'Country'})
    gdp_growth = gdp_growth[['Country Name', '2015', '2016', '2017', '2018', '2019']].rename(columns={'Country Name': 'Country'})
    
    # Rename all yearly columns to GDP_Per_Capita_x
    gdp = gdp.rename(columns={
        '2015': 'GDP_Per_Capita_2015',
        '2016': 'GDP_Per_Capita_2016',
        '2017': 'GDP_Per_Capita_2017',
        '2018': 'GDP_Per_Capita_2018',
        '2019': 'GDP_Per_Capita_2019'
    })
    
    # Rename all yearly columns to GDP_Per_Capita_Growth_x
    gdp_growth = gdp_growth.rename(columns={
        '2015': 'GDP_Per_Capita_Growth_2015',
        '2016': 'GDP_Per_Capita_Growth_2016',
        '2017': 'GDP_Per_Capita_Growth_2017',
        '2018': 'GDP_Per_Capita_Growth_2018',
        '2019': 'GDP_Per_Capita_Growth_2019'
    })
    
    # Merge the above datasets on the Country column
    gdp_data = pd.merge(gdp, gdp_growth, on='Country', how='inner')
    
    # Compute the mean GDP per capita across the years and add as a new column
    gdp_per_capita_cols = [col for col in gdp_data.columns if 'GDP_Per_Capita_20' in col]
    gdp_data['GDP_Per_Capita_Mean'] = gdp_data[gdp_per_capita_cols].mean(axis=1)
    
    # Compute the mean GDP per capita growth across the years and add as a new column
    gdp_per_capita_growth_cols = [col for col in gdp_data.columns if 'GDP_Per_Capita_Growth_20' in col]
    gdp_data['GDP_Per_Capita_Growth_Mean'] = gdp_data[gdp_per_capita_growth_cols].mean(axis=1)
    
    # Save the final dataset
    gdp_data.to_csv('data/gdp_data.csv', index=False)
else:
    gdp_data = pd.read_csv('data/gdp_data.csv')

## 3. Combining Data
Combine the datasets into a single dataframe.

In [39]:
# Combine all datasets
if not Path('data/combined_data.csv').exists():
    #load all the datasets
    if not(Path('data/happiness_data.csv').exists() and Path('data/unemployment_data.csv').exists() and Path('data/gdp_data.csv').exists()):
        raise Exception('One or more of the datasets is missing, please make sure all datasets are present in the data folder. If not rerun the scripts')
    happiness_data = pd.read_csv('data/happiness_data.csv')
    unemployment_data = pd.read_csv('data/unemployment_data.csv')
    gdp_data = pd.read_csv('data/gdp_data.csv')
    #merge the above datasets on the country column
    combined_data = pd.merge(happiness_data, unemployment_data, on='Country', how='inner')
    combined_data = pd.merge(combined_data, gdp_data, on='Country', how='inner')

    #save the final dataset
    combined_data.to_csv('data/combined_data.csv', index=False)
else:
    combined_data = pd.read_csv('data/combined_data.csv')
    print('Data already combined')